# TrainWise — What-If Training-Load Planner (#185)

**Python ML course project · Scenario analysis on the Acute:Chronic Workload Ratio**

This notebook is the data-science write-up behind the **What-if planner** card on the coach
*Analytics* screen and the trainee *My analytics* screen. The planner answers a forward-looking
question a coach actually asks:

> *“If I add **N** more sessions this week at an easy / medium / hard intensity, where does this
> athlete's injury-risk ratio land?”*

It is a **simulation / sensitivity-analysis** layer on top of the same ACWR model built in
`TrainWise_Load_Analytics.ipynb`. Instead of only *measuring* load, we *perturb* it and re-measure,
so the coach can plan a safe progression **before** the athlete trains.

## Background — why this works

$$\text{ACWR} = \frac{\text{acute load (last 7 days)}}{\text{chronic load (last 28 days, weekly avg)}}$$

Adding sessions **this week** pushes load onto the *acute* (7-day) window immediately, while the
*chronic* (28-day) average barely moves — a few sessions are a small fraction of 28 days. So the
ratio climbs **mostly through the numerator**, which is exactly the real physiological effect ACWR is
designed to capture. The planner lets you see the risk pill flip from green → amber → **red**
*before* it happens on the body.

- **ACWR < 0.8** — detraining · **0.8–1.3** sweet spot · **> 1.3** spiking · **> 1.5** danger.
- A **session load** unit = `duration (min) × exertion (RPE 1-10)`. The planner models three
  intensities as fixed loads: **easy = 150, medium = 300, hard = 450** (e.g. medium ≈ 50 min @ RPE 6).

## Mapping to the course tasks

| Course task | Here |
|---|---|
| Data cleaning + EDA | §2 — synthetic athlete histories, load distribution |
| Feature engineering | §3 — daily series + rolling ACWR (cold-start floor + covered-days ramp) |
| **Scenario / what-if simulation** | §4–§5 — inject N sessions, recompute, sweep the response curve |
| **Regression** (Task 1 link) | §6 — a linear surrogate that predicts simulated ACWR from (chronic, added load) |
| Model insight | §7 — “safe headroom”: how many sessions until the athlete leaves the sweet spot |

> The **live** service is `ml/forecast.py::simulate_whatif`, exposed as
> `GET /api/ml/trainee/<id>/whatif?addSessions=&intensity=`. The formulas below are the **same**
> rolling-ACWR guards used there and in `ml/features.py` / `utils/acwr.js`, so the curves in this
> notebook match what the app shows. Here we use reproducible synthetic data so it runs standalone.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

sns.set_theme(style="whitegrid", context="notebook")
RNG = np.random.default_rng(42)          # reproducible
pd.set_option("display.width", 120)

# The three what-if intensities, IDENTICAL to ml/forecast.py::WHATIF_SESSION_LOAD.
WHATIF_SESSION_LOAD = {"easy": 150.0, "medium": 300.0, "hard": 450.0}
WHATIF_MAX_SESSIONS = 14                  # server-side clamp (also debounced client-side)
BOOTSTRAP = {1: 150.0, 2: 280.0, 3: 420.0}  # experience -> weekly acute floor (LoadParameters seed)
print("Environment ready. Intensities:", WHATIF_SESSION_LOAD)

## 2. Data — a synthetic training history

We simulate one **Regular** athlete over the last 28 days (the chronic window), because the planner
only needs the trailing window ending *today* to compute the current state. The columns mirror the
app's `ActivityLogs`: a per-day `load` (= duration × RPE), zero on rest days.

In [ ]:
N_DAYS = 28
EXP = 2                                   # Regular
START = pd.Timestamp.today().normalize() - pd.Timedelta(days=N_DAYS - 1)
idx = pd.date_range(START, periods=N_DAYS, freq="D")

# ~4 training days/week, moderate RPE, a couple of harder days.
loads = []
for d in range(N_DAYS):
    train = RNG.random() < 4 / 7
    if not train:
        loads.append(0.0)
        continue
    dur = int(np.clip(RNG.normal(45, 12), 20, 90))
    rpe = int(np.clip(RNG.normal(5.5, 1.4), 2, 9))
    loads.append(float(dur * rpe))

daily = pd.Series(loads, index=idx, name="load")
print(f"Active days: {(daily > 0).sum()} / {N_DAYS}  |  28-day total load: {daily.sum():.0f}")
daily.tail(10)

In [ ]:
plt.figure(figsize=(13, 3))
plt.bar(daily.index, daily.values, width=0.9, color="#457b9d")
plt.title("Daily session load — trailing 28 days (the window the planner works from)")
plt.ylabel("load = duration × RPE"); plt.tight_layout(); plt.show()

## 3. Rolling ACWR with the app's guards

Exactly the function from `TrainWise_Load_Analytics.ipynb` (and `ml/features.rolling_loads` /
`utils/acwr.js`): acute = trailing 7-day sum; chronic = 28-day load on a weekly scale, with the two
guards that make it correct for short histories:

1. **Cold-start floor** — with < 7 active days in the window, floor chronic at the experience
   bootstrap (150 / 280 / 420 weekly).
2. **Covered-days ramp** — once ≥ 7 active days, divide by the weeks actually covered
   (`sum28 / min(4, covered/7)`), so a 2-week-old athlete reads ~1.0, not a false 2.0.

We only need the state on the **last** day (today), so a small helper returns acute / chronic / ratio
at the end of a daily series.

In [ ]:
ACUTE_W, CHRONIC_W, DIVISOR = 7, 28, 4.0

def rolling_acwr(daily, exp):
    acute = daily.rolling(ACUTE_W, min_periods=1).sum()
    sum28 = daily.rolling(CHRONIC_W, min_periods=1).sum()
    active = (daily > 0).rolling(CHRONIC_W, min_periods=1).sum().to_numpy()
    boot = BOOTSTRAP[exp]
    vals = daily.to_numpy(); n = len(vals); s28 = sum28.to_numpy()
    floored = np.maximum(s28 / DIVISOR, boot)                 # guard 1: cold-start floor
    chronic = floored.copy()
    nz = np.flatnonzero(vals > 0)
    if nz.size:                                               # guard 2: covered-days ramp
        idxs = np.arange(n)
        j = np.searchsorted(nz, idxs - (CHRONIC_W - 1), side="left")
        first = nz[np.minimum(j, nz.size - 1)]
        covered = np.maximum(idxs - first + 1, 7)
        ramped = s28 / np.minimum(DIVISOR, covered / 7.0)
        chronic = np.where(active >= 7, ramped, floored)
    ratio = np.where(chronic > 0, acute.to_numpy() / chronic, np.nan)
    return pd.DataFrame({"acute": acute, "chronic": chronic, "acwr": ratio}, index=daily.index)

def state_today(daily, exp):
    """acute / chronic / AC ratio on the LAST day - mirrors forecast._state_from_rolled."""
    row = rolling_acwr(daily, exp).iloc[-1]
    return {"acute": round(float(row.acute)), "chronic": round(float(row.chronic)),
            "acRatio": round(float(row.acwr), 2)}

def risk_band(ratio, has_injury=False):
    """Same thresholds as risk.rule_class / determineLevel (injured tightens to 1.2)."""
    hi = 1.2 if has_injury else 1.3
    if ratio > 1.5: return "High"
    if ratio > hi:  return "High"
    if ratio >= 0.8: return "Warning"
    return "Safe"

base = state_today(daily, EXP)
print("Baseline today:", base, "-> risk:", risk_band(base["acRatio"]))

## 4. The what-if simulation

This is the notebook twin of `ml/forecast.py::simulate_whatif`. We add `add_sessions × session_load`
onto **today**, then recompute with the *same* `rolling_acwr`. Placing all the added load on today is
deliberate: acute is a rolling 7-day **sum**, so N sessions today give the same weekly acute as
spreading them across the week, while keeping the comparison at a single time point so the delta
isolates exactly the sessions the coach dialed in. Inputs are clamped, just like the server.

In [ ]:
def simulate_whatif(daily, exp, add_sessions, intensity, has_injury=False):
    add_sessions = max(0, min(int(add_sessions), WHATIF_MAX_SESSIONS))          # clamp
    intensity = intensity if intensity in WHATIF_SESSION_LOAD else "medium"
    session_load = WHATIF_SESSION_LOAD[intensity]

    baseline = state_today(daily, exp)
    daily_wi = daily.copy()
    daily_wi.iloc[-1] = float(daily_wi.iloc[-1]) + add_sessions * session_load   # inject onto today
    sim = state_today(daily_wi, exp)
    return {
        "addSessions": add_sessions, "intensity": intensity,
        "addedLoad": add_sessions * session_load, "hasInjury": has_injury,
        "baseline": {**baseline, "risk": risk_band(baseline["acRatio"], has_injury)},
        "simulated": {**sim, "risk": risk_band(sim["acRatio"], has_injury)},
    }

# One scenario, exactly what the app card renders: +3 hard sessions.
from pprint import pprint
pprint(simulate_whatif(daily, EXP, add_sessions=3, intensity="hard"))

## 5. The response curve — sweeping the slider

The planner's slider goes 0→N. Here we sweep it for all three intensities and plot the resulting AC
ratio against the sweet-spot band, so you can literally read off *how many* sessions of each
intensity take the athlete to the edge of green, into amber, or over the red line.

In [ ]:
adds = np.arange(0, WHATIF_MAX_SESSIONS + 1)
curves = {
    inten: [simulate_whatif(daily, EXP, n, inten)["simulated"]["acRatio"] for n in adds]
    for inten in WHATIF_SESSION_LOAD
}

fig, ax = plt.subplots(figsize=(12, 5))
ax.axhspan(0.8, 1.3, color="#2a9d8f", alpha=0.12, label="sweet spot 0.8–1.3")
ax.axhline(1.3, color="#e9c46a", ls="--", lw=1)
ax.axhline(1.5, color="#e76f51", ls="--", lw=1, label="danger 1.5")
colors = {"easy": "#2a9d8f", "medium": "#e9c46a", "hard": "#e76f51"}
for inten, ys in curves.items():
    ax.plot(adds, ys, marker="o", color=colors[inten], lw=2,
            label=f"{inten} (+{int(WHATIF_SESSION_LOAD[inten])}/session)")
ax.scatter([0], [curves['hard'][0]], color="black", zorder=6)
ax.annotate("today", (0, curves['hard'][0]), textcoords="offset points", xytext=(6, 8))
ax.set_xlabel("sessions added this week"); ax.set_ylabel("projected AC ratio")
ax.set_title("What-if: AC ratio vs sessions added, by intensity")
ax.legend(loc="upper left"); plt.tight_layout(); plt.show()

In [ ]:
# The same data as a decision table: the risk band after each added session.
tbl = pd.DataFrame({inten: [risk_band(r) for r in ys] for inten, ys in curves.items()}, index=adds)
tbl.index.name = "add"
tbl.head(9)

## 6. A linear surrogate (Task 1 link)

The response is close to linear in *added load* for a fixed chronic (acute rises linearly, chronic
moves little), so a tiny **regression** recovers the relationship and lets us reason about it
analytically. We fit `AC ratio ~ added_load` across the swept scenarios and report MAE / R² — a
compact demonstration of the course's regression task on the planner's own output.

In [ ]:
rows = []
for inten in WHATIF_SESSION_LOAD:
    for n in adds:
        r = simulate_whatif(daily, EXP, n, inten)
        rows.append({"added_load": r["addedLoad"], "acwr": r["simulated"]["acRatio"]})
sweep = pd.DataFrame(rows)

Xr = sweep[["added_load"]].to_numpy(); yr = sweep["acwr"].to_numpy()
reg = LinearRegression().fit(Xr, yr)
pred = reg.predict(Xr)
print(f"AC ratio ~ {reg.intercept_:.3f} + {reg.coef_[0]:.5f} * added_load")
print(f"MAE={mean_absolute_error(yr, pred):.3f}  R2={r2_score(yr, pred):.3f}")

# Slope interpretation: load needed to raise ACWR by 0.1, and headroom before the 1.3 line.
load_per_0_1 = 0.1 / reg.coef_[0]
headroom_load = max(0.0, (1.3 - reg.intercept_) / reg.coef_[0])
print(f"~{load_per_0_1:.0f} load units raise ACWR by 0.10")
print(f"~{headroom_load:.0f} load units of headroom before ACWR 1.3",
      f"(about {headroom_load / 300:.1f} medium sessions)")

## 7. “Safe headroom” — how the advice depends on current fitness

The most useful planning number is: **how many medium sessions can this athlete add before leaving
the sweet spot (ACWR 1.3)?** That headroom depends on their *chronic* base — a fitter athlete
(higher chronic) can absorb more. We sweep a range of chronic levels and compute the max safe
sessions, which is the intuition the planner surfaces to the coach.

In [ ]:
def max_safe_sessions(daily, exp, intensity, limit=1.3):
    for n in range(0, WHATIF_MAX_SESSIONS + 1):
        if simulate_whatif(daily, exp, n, intensity)["simulated"]["acRatio"] > limit:
            return max(0, n - 1)
    return WHATIF_MAX_SESSIONS

# Scale the whole history up/down to emulate lower/higher chronic bases.
scales = np.linspace(0.5, 2.0, 16)
headroom = []
for s in scales:
    d = daily * s
    chronic = state_today(d, EXP)["chronic"]
    headroom.append({"chronic": chronic, "safe_medium": max_safe_sessions(d, EXP, "medium")})
hr = pd.DataFrame(headroom)

plt.figure(figsize=(11, 4))
plt.plot(hr["chronic"], hr["safe_medium"], marker="o", color="#264653", lw=2)
plt.xlabel("chronic load (fitness base)"); plt.ylabel("safe medium sessions to add")
plt.title("Safe headroom before ACWR 1.3 rises with the athlete's chronic base")
plt.tight_layout(); plt.show()
hr.head()

## 8. Conclusion

- The **What-if planner** is a scenario layer on the ACWR model: it injects `N × intensity_load` onto
  today and recomputes acute / chronic / ratio with the **same** cold-start-floored, covered-days-ramped
  rolling ACWR the rest of TrainWise uses — so a projected number never contradicts the app.
- The response curve (§5) turns an abstract ratio into a concrete plan: *“≤ 3 medium sessions keeps you
  green; 5 pushes into amber; hard sessions get there twice as fast.”*
- A linear surrogate (§6) and the **safe-headroom** sweep (§7) show the coaching intuition analytically:
  headroom scales with the athlete's chronic base, which is exactly why a fitter athlete can ramp faster.

**In the app:** this is `ml/forecast.py::simulate_whatif`, served by
`GET /api/ml/trainee/<id>/whatif?addSessions=&intensity=easy|medium|hard`, and rendered as the
debounced intensity + slider **What-if planner** card on the coach *Analytics* and trainee
*My analytics* screens (returning `{baseline, simulated}` with matching risk pills).